### PDF 版面解析实验

解析 `data/corpus/kb_pdf/pdf/社区服务指南 第9部分：物业服务.pdf`：

1. **PyMuPDF** 按坐标提取文本块（字号、位置）
2. 去掉页眉 / 页脚
3. 按字号识别标题，切成章节
4. **质量检测**：若几乎抽不出中文，说明 PDF 字体未映射 Unicode，需 OCR

> 首次运行若缺依赖：`%pip install pymupdf`

In [ ]:
from __future__ import annotations

import re
from dataclasses import dataclass, field
from pathlib import Path

import fitz  # PyMuPDF  (%pip install pymupdf)

# 项目根目录：notebook 在 src/ipynb/
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent.parent / "data").exists():
    ROOT = ROOT.parent.parent
elif not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

PDF_PATH = ROOT / "data/corpus/kb_pdf/pdf/社区服务指南 第9部分：物业服务.pdf"
assert PDF_PATH.exists(), f"找不到文件: {PDF_PATH}"

print("PDF:", PDF_PATH)
print("大小:", f"{PDF_PATH.stat().st_size / 1024:.1f} KB")

In [ ]:
@dataclass
class TextSpan:
    text: str
    page: int
    x0: float
    y0: float
    x1: float
    y1: float
    size: float
    font: str

    @property
    def mid_y(self) -> float:
        return (self.y0 + self.y1) / 2


@dataclass
class Section:
    title: str
    page_start: int
    spans: list[TextSpan] = field(default_factory=list)

    def body_text(self) -> str:
        lines: list[str] = []
        prev_y: float | None = None
        buf: list[str] = []
        for sp in sorted(self.spans, key=lambda s: (s.page, s.y0, s.x0)):
            if prev_y is not None and abs(sp.y0 - prev_y) > 4:
                if buf:
                    lines.append("".join(buf).strip())
                    buf = []
            buf.append(sp.text)
            prev_y = sp.y0
        if buf:
            lines.append("".join(buf).strip())
        return "\n".join(ln for ln in lines if ln)


def cjk_ratio(text: str) -> float:
    if not text:
        return 0.0
    cjk = len(re.findall(r"[\u4e00-\u9fff]", text))
    return cjk / max(len(text.replace(" ", "").replace("\n", "")), 1)


def extract_spans(doc: fitz.Document) -> list[TextSpan]:
    """从每页 dict 模式提取带坐标的 span。"""
    spans: list[TextSpan] = []
    for page_index in range(doc.page_count):
        page = doc[page_index]
        for block in page.get_text("dict").get("blocks", []):
            if block.get("type") != 0:
                continue
            for line in block.get("lines", []):
                for sp in line.get("spans", []):
                    text = (sp.get("text") or "").strip()
                    if not text:
                        continue
                    bbox = sp["bbox"]
                    spans.append(
                        TextSpan(
                            text=text,
                            page=page_index,
                            x0=bbox[0],
                            y0=bbox[1],
                            x1=bbox[2],
                            y1=bbox[3],
                            size=round(float(sp.get("size", 0)), 2),
                            font=str(sp.get("font", "")),
                        )
                    )
    return spans


def drop_header_footer(
    spans: list[TextSpan],
    doc: fitz.Document,
    *,
    margin_ratio: float = 0.08,
) -> list[TextSpan]:
    """按页高裁掉顶部/底部区域；并去掉跨页重复的短行（常见页眉脚）。"""
    by_page: dict[int, list[TextSpan]] = {}
    for sp in spans:
        by_page.setdefault(sp.page, []).append(sp)

    top_lines: list[str] = []
    bottom_lines: list[str] = []
    for page_index, page_spans in by_page.items():
        h = doc[page_index].rect.height
        top_cut = h * margin_ratio
        bot_cut = h * (1 - margin_ratio)
        top_zone = [s for s in page_spans if s.mid_y < top_cut]
        bot_zone = [s for s in page_spans if s.mid_y > bot_cut]
        if top_zone:
            top_lines.append("".join(s.text for s in sorted(top_zone, key=lambda x: x.x0)))
        if bot_zone:
            bottom_lines.append("".join(s.text for s in sorted(bot_zone, key=lambda x: x.x0)))

    def repeated_short(lines: list[str], min_pages: int = 2) -> set[str]:
        from collections import Counter

        c = Counter(ln.strip() for ln in lines if 0 < len(ln.strip()) <= 40)
        return {ln for ln, n in c.items() if n >= min_pages}

    drop_texts = repeated_short(top_lines) | repeated_short(bottom_lines)
    page_re = re.compile(r"^[-–]?\s*\d+\s*[-–]?$")

    kept: list[TextSpan] = []
    for sp in spans:
        h = doc[sp.page].rect.height
        top_cut = h * margin_ratio
        bot_cut = h * (1 - margin_ratio)
        if sp.mid_y < top_cut or sp.mid_y > bot_cut:
            continue
        if sp.text.strip() in drop_texts:
            continue
        if page_re.fullmatch(sp.text.strip()):
            continue
        kept.append(sp)
    return kept


def split_sections_by_font(
    spans: list[TextSpan],
    *,
    body_size: float | None = None,
    heading_ratio: float = 1.15,
) -> list[Section]:
    """无 PDF 书签时：字号明显大于正文的 span 当作标题，开新 section。"""
    if not spans:
        return []

    sizes = sorted(s.size for s in spans)
    if body_size is None:
        # 正文字号：取中位数附近
        body_size = sizes[len(sizes) // 2]

    heading_threshold = body_size * heading_ratio
    sections: list[Section] = []
    current = Section(title="(文档开头)", page_start=spans[0].page)

    for sp in spans:
        is_heading = sp.size >= heading_threshold and len(sp.text) <= 60
        if is_heading and current.spans:
            sections.append(current)
            current = Section(title=sp.text, page_start=sp.page)
        else:
            if is_heading and not current.spans and current.title == "(文档开头)":
                current.title = sp.text
            else:
                current.spans.append(sp)
    sections.append(current)
    return sections


def split_sections_by_toc(doc: fitz.Document, spans: list[TextSpan]) -> list[Section] | None:
    """有 PDF 书签（outline）时按页码切 section（最稳）。"""
    toc = doc.get_toc(simple=True)
    if not toc:
        return None

    ranges: list[tuple[str, int, int]] = []
    for i, (_level, title, page1) in enumerate(toc):
        start = max(page1 - 1, 0)
        end = doc.page_count - 1 if i + 1 >= len(toc) else max(toc[i + 1][2] - 2, start)
        ranges.append((title.strip(), start, end))

    sections: list[Section] = []
    for title, start, end in ranges:
        part = [s for s in spans if start <= s.page <= end]
        sections.append(Section(title=title, page_start=start, spans=part))
    return sections


def parse_pdf_layout(
    path: Path,
    *,
    margin_ratio: float = 0.08,
) -> dict:
    doc = fitz.open(path)
    raw_spans = extract_spans(doc)
    clean_spans = drop_header_footer(raw_spans, doc, margin_ratio=margin_ratio)

    sections = split_sections_by_toc(doc, clean_spans)
    mode = "toc"
    if sections is None:
        sections = split_sections_by_font(clean_spans)
        mode = "font"

    full_text = "\n\n".join(
        f"## {sec.title}\n{sec.body_text()}" for sec in sections if sec.body_text().strip()
    )

    result = {
        "path": str(path),
        "page_count": doc.page_count,
        "toc_entries": len(doc.get_toc(simple=True)),
        "raw_span_count": len(raw_spans),
        "clean_span_count": len(clean_spans),
        "section_mode": mode,
        "sections": sections,
        "full_text": full_text,
        "cjk_ratio": cjk_ratio(full_text),
    }
    doc.close()
    return result

In [ ]:
result = parse_pdf_layout(PDF_PATH)

print(f"页数: {result['page_count']}")
print(f"书签条目: {result['toc_entries']}")
print(f"切块模式: {result['section_mode']}")
print(f"原始 span: {result['raw_span_count']} → 清洗后: {result['clean_span_count']}")
print(f"章节数: {len(result['sections'])}")
print(f"中文占比 (CJK ratio): {result['cjk_ratio']:.2%}")

if result["cjk_ratio"] < 0.05:
    print("\n⚠️  几乎抽不出中文：该 PDF 可能使用了未映射 Unicode 的嵌入字体，")
    print("   版面解析流程仍可用，但正文是乱码 → 需要 OCR（见下一格）。")
else:
    print("\n✅ 文本抽取正常，可直接用于切块入库。")

print("\n--- 章节列表 ---")
for i, sec in enumerate(result["sections"][:12], 1):
    preview = sec.body_text().replace("\n", " ")[:80]
    print(f"{i:2}. p{sec.page_start + 1:02} | {sec.title[:40]} | {preview}...")

In [ ]:
# 查看某一章完整正文（改 index 即可）
SECTION_INDEX = 0
sec = result["sections"][SECTION_INDEX]
print(f"# {sec.title}  (第 {sec.page_start + 1} 页起)\n")
print(sec.body_text()[:2000])

In [ ]:
def ocr_pdf(path: Path, *, lang: str = "chi_sim") -> str:
    """扫描件 / 字体乱码 PDF：渲染为图 + Tesseract OCR。

    需本机安装：
      brew install tesseract tesseract-lang
    """
    import shutil

    if not shutil.which("tesseract"):
        raise RuntimeError("未找到 tesseract，请先: brew install tesseract tesseract-lang")

    doc = fitz.open(path)
    parts: list[str] = []
    for i in range(doc.page_count):
        page = doc[i]
        # PyMuPDF 内置 OCR 封装（依赖系统 tesseract）
        tp = page.get_textpage_ocr(flags=0, language=lang, dpi=200, full=True)
        text = page.get_text(textpage=tp)
        parts.append(f"<!-- page {i + 1} -->\n{text.strip()}")
    doc.close()
    return "\n\n".join(parts)


# 仅当 CJK 过低时尝试 OCR（取消注释运行）
# ocr_text = ocr_pdf(PDF_PATH)
# print("OCR 中文占比:", f"{cjk_ratio(ocr_text):.2%}")
# print(ocr_text[:1500])

In [ ]:
# 可选：把解析结果导出为 Markdown，便于人工检查 / 转 policies 语料
OUT_MD = ROOT / "data/corpus/kb_pdf/pdf/_parsed_物业服务.md"

if result["cjk_ratio"] >= 0.05:
    OUT_MD.write_text(result["full_text"], encoding="utf-8")
    print("已写入:", OUT_MD)
else:
    print("当前为乱码文本，跳过写入。请先 OCR 或换源文件（带文字层的 PDF）。")